## Entities

In [5]:
BASE_PROMPT="""
You are an expert SPARK SQL semantic modeler. Transform raw SPARK SQL into structured JSON for data modeling. Output ONLY valid JSON matching this exact schema:

{
  "entities": {
    "<BASE_entity_name>": {
      "alias": "<table_alias>",
      "fields": {
        "<field_name>": {}, ...
      }
    }, ...
    "<CTE_entity_name>": {
      "alias": "<table_alias>",
      "fields": {
        "<field_name>": {
          "ref": ["<(BASE/CTE)_entity_name>.<field_name>", "..."],
          "calculation": {
            "expression": "<calculation_expression_sql>",
            "ref": ["<(BASE/CTE)_entity_name>.<field_name>", "..."]
          }, ...
        }
      }
    }, ...
    "<VIEW_entity_name>": {
      "alias": "<table_alias>",
      "fields": {
        "<field_name>": {
          "ref": ["<(BASE/CTE)_entity_name>.<field_name>", "..."],
          "calculation": {
            "expression": "<calculation_expression_sql>",
            "ref": ["<(BASE/CTE)_entity_name>.<field_name>", "..."]
          }
        }, ...
      }
    }
  }
}

=== STRICT RULES ===
1. BASE: Base tables (FROM/JOIN), which have no further references to other tables.
2. CTE: WITH clause entities.
3. VIEW: Final SELECT;
5. Put PREFIXES such as BASE, CTE and VIEW for each entities accordingly in the given format:
4. **Order of references**: Base <- CTE <- Final View / CTE <- Final View / Base <- Final View.
5  **Field_name**: Check in select, from, join, where, having for fields.
6. **Base Tables**:
   - `alias`: if present, otherwise omit.
   - `field_name`: `{}` only. No `ref`, no `calculation`.
7. **CTE & View**:
   - `alias`: (before `AS`) if present, otherwise omit.
   - `calculation`: Add if computed → `{ "expression": "...", "ref": [...] }`.
   - Omit `ref` or `calculation` if not applicable.
8. **Expression**: Exact SQL snippet using aliases.
9. Group base tables first, then CTEs/views.
10. Dont put * as field_name. Expand * into individual fields from the referenced entity accordingly.
11. If referencing is done in ref, make sure that the referenced field exists in the referenced entity also.
12. **Output**: JSON only. No extras.
"""

In [ ]:
file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\AccountsPayable.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\AccountsPayableOverview.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\AccountsPayableTurnover.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\BalanceSheet.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\CashDiscountUtilization.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\InventoryKeyMetrics.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\SalesFulfillment.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\VendorLeadTimeOverview.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\VendorPerformance.sql"
# file_to_read = r"C:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\VendorPerformanceOverview.sql"

try:
    with open(file_to_read, "r", encoding="utf-8", errors="replace") as f:
        file_content = f.read()
except FileNotFoundError:
    print(f"File not found: {file_to_read}")
else:
    filled_prompt = BASE_PROMPT + "\nSPARK SQL CONTENT: \n---\n" + file_content + "\n---\n"
    print(filled_prompt)

## Joins

In [38]:
BASE_PROMPT = """
Extract join details for all CTEs and FINAL SELECT statements from the SQL query above.

- BASE tables: Tables with no further references
- CTE tables: Tables created by WITH clause
- VIEW tables: Final SELECT clauses

Naming Convention:
- Prefix all table names with BASE_, CTE_, or VIEW_ accordingly

Join Extraction Rules:
1. If a CTE or VIEW has no joins, use an empty array []
2. For each join condition:
   - "calculation": Include the full expression only if the join condition involves any operations otherwise omit this field
   - "type": The join type

JSON Format:
{
  "join": {
    "(CTE/VIEW)_table_name": [
      {
        "from": {
          "field": "BASE_table_name.field_name", // strictly field name only, no calculations
          "calculation": "optional - full expression if calculation exists"
        },
        "to": {
          "field": "CTE_table_name.field_name", // strictly field name only, no calculations
          "calculation": "optional - full expression if calculation exists"
        },
        "type": "LEFT"
      }
    ], ...
  }
}

Note:
- Handle multiple join conditions (AND/OR) as separate join objects
"""

In [ ]:
files = [
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\AccountsPayable.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\AccountsPayableOverview.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\AccountsPayableTurnover.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\BalanceSheet.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\CashDiscountUtilization.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\InventoryKeyMetrics.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\SalesFulfillment.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\VendorLeadTimeOverview.sql",
    # r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\VendorPerformance.sql",
    r"c:\Users\gladwin.aj\Downloads\DnA\Data Model\SQL_Data_Model\src\testing\SQL\VendorPerformanceOverview.sql"
]

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        content = f.read()
    print(content + BASE_PROMPT)